# GUI-Model: Mobile UI Transition & Action Predictor

Qwen3-VL-8B-Instruct 기반 모바일 UI 예측 모델 Fine-tuning 파이프라인.

**전체 파이프라인:**
1. **Stage 1: World Modeling** - UI State (XML) + Action → Next UI State (XML)
2. **Stage 2: Action Prediction** - Screenshot + UI State + Task → Action (JSON)
3. **Evaluation** - Test set으로 3-Way 모델 비교 평가
4. **Merge & Upload** - Best 모델 Merge 후 HuggingFace Hub 업로드

**모델 정보:**

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Template | qwen3_vl_nothink |
| Vision Tower | Frozen |

**데이터셋 정보:**

| Dataset | Stage | Entries | Description |
|---------|-------|---------|-------------|
| GUI-Model_stage1_train | 1 | 3,087 | 전체 (OpenApp 포함) |
| GUI-Model_stage1_NoOpenApp_train | 1 | 2,073 | OpenApp 제외 |
| GUI-Model_stage2_train | 2 | ~3,290 | Action Prediction (Train 90%) |
| GUI-Model_stage2_test | 2 | ~365 | Action Prediction (Test 10%) |
| Images | - | 3,655 | Mobile UI Screenshots (PNG, 공유) |

**Stage 1 학습 실험:**

| ID | Method | LR Schedule | Learning Rate | Config |
|----|--------|-------------|---------------|--------|
| [1] | Full FT | constant | 2e-5 | stage1_full/qwen3_vl_8b_gui.yaml |

**Stage 2 학습 실험 (3-Way 비교):**

| ID | Base Model | HuggingFace ID | Config |
|----|------------|----------------|--------|
| [S2-1] | Qwen3-VL-8B (Baseline) | `Qwen/Qwen3-VL-8B-Instruct` | stage2_lora/baseline.yaml |
| [S2-2] | Code2World-8B | `GD-ML/Code2World` | stage2_lora/code2world.yaml |
| [S2-3] | gWorld-8B | `trillionlabs/gWorld-8B` | stage2_lora/gworld.yaml |

**Stage 2 공통 설정:** LoRA r=16, α=32, dropout=0.1, LR=5e-5 (cosine), batch=4x2x2GPU=16, A100x2

**평가 메트릭:**

| Metric | Description | Applicable Types |
|--------|-------------|-----------------|
| Type Accuracy | Action type 일치율 | All |
| Bounds IoU | Bounding box IoU | Click, Input, Swipe, Long Click |
| Params Match | 파라미터 정확도 | OpenApp(app), Input(text), Swipe(direction) |
| Overall Score | Type_Acc × (0.5×IoU + 0.5×Params) | All |

## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive/')

In [ ]:
import os

# ============================================================
# === 여기서 프로젝트 루트 경로를 설정하세요 ===
BASE_DIR = "/path/to/GUI-Model"
# ============================================================

LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")
DATA_DIR = os.path.join(BASE_DIR, "data")

os.chdir(BASE_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/hiyouga/LlamaFactory.git

In [ ]:
os.chdir(LF_ROOT)
!pip install -e ".[torch,metrics]"

## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 LLaMA-Factory에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: 3,655개 모바일 UI 스크린샷

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage1.jsonl | 3,087 | 전체 (OpenApp 포함) |
| gui-model_stage1_NoOpenApp.jsonl | 2,073 | OpenApp 액션 제외 |

In [ ]:
import json
import shutil
from pathlib import Path

# === Paths ===
DATA_PATH = Path(DATA_DIR)
LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. 데이터 디렉토리 생성 ===
LF_GUI_DATA.mkdir(parents=True, exist_ok=True)
(LF_GUI_DATA / "images").mkdir(exist_ok=True)

# === 2. JSONL 파일 복사 ===
for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_NoOpenApp.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"[Copy] {fname} ({src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 3. 이미지 복사 ===
src_images = DATA_PATH / "images"
dst_images = LF_GUI_DATA / "images"

if src_images.exists():
    img_count = len(list(src_images.glob("*.png")))
    existing = len(list(dst_images.glob("*.png")))
    if existing >= img_count:
        print(f"[Skip] Images already exist ({existing} files)")
    else:
        for img in src_images.glob("*.png"):
            dst_img = dst_images / img.name
            if not dst_img.exists():
                shutil.copy2(img, dst_img)
        print(f"[Copy] {img_count} images")
else:
    print(f"[WARN] Image directory not found: {src_images}")

# === 4. dataset_info.json 등록 ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

DATASETS_TO_REGISTER = {
    "GUI-Model_stage1_train": {
        "file_name": "GUI-Model/gui-model_stage1.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system"
        }
    },
    "GUI-Model_stage1_NoOpenApp_train": {
        "file_name": "GUI-Model/gui-model_stage1_NoOpenApp.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
            "system_tag": "system"
        }
    }
}

for name, config in DATASETS_TO_REGISTER.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 1 Dataset Statistics ===\n")

for fname in ["gui-model_stage1.jsonl", "gui-model_stage1_NoOpenApp.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()
        entry_count = len(lines)

        # 첫 번째 엔트리 샘플
        sample = json.loads(lines[0])
        msg_count = len(sample["messages"])
        img_count = len(sample.get("images", []))

        print(f"[{fname}]")
        print(f"  Entries: {entry_count}")
        print(f"  Sample - Messages: {msg_count}, Images: {img_count}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
        print()

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 1.5 Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 변환, **Train/Test Split**, LLaMA-Factory 등록을 수행합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1과 동일한 3,655개 (공유)

| File | Entries | Description |
|------|---------|-------------|
| gui-model_stage2.jsonl | 3,655 | Action Prediction (원본) |
| gui-model_stage2_train.jsonl | ~3,290 | Train Split (90%) |
| gui-model_stage2_test.jsonl | ~365 | Test Split (10%) |

**변환 작업:**
- 원본의 `id`, `assets`, `source` 필드 제거
- `images` 필드 추가: `["GUI-Model/images/{id}.png"]`

**Split 전략:**
- Stratified Split (action type 비율 유지)
- Random seed: 42 (재현 가능)

In [ ]:
import json
from pathlib import Path

stage2_path = Path(DATA_DIR) / "gui-model_stage2.jsonl"

# === 1. 변환 필요 여부 확인 ===
with open(stage2_path, 'r') as f:
    first = json.loads(f.readline())

if "images" in first and "id" not in first:
    print("[Skip] Stage 2 data already converted")
    print(f"  Keys: {list(first.keys())}")
    print(f"  Images: {first['images']}")
else:
    # === 2. 변환 수행 ===
    with open(stage2_path, 'r') as f:
        lines = f.readlines()

    print(f"[Before] {len(lines)} entries, Keys: {list(first.keys())}")

    converted = []
    for line in lines:
        entry = json.loads(line)
        new_entry = {
            "messages": entry["messages"],
            "images": [f"GUI-Model/images/{entry['id']}.png"]
        }
        converted.append(json.dumps(new_entry, ensure_ascii=False))

    with open(stage2_path, 'w') as f:
        f.write('\n'.join(converted) + '\n')

    # === 3. 검증 ===
    with open(stage2_path, 'r') as f:
        check = json.loads(f.readline())

    print(f"[After] {len(converted)} entries, Keys: {list(check.keys())}")
    print(f"  Sample images: {check['images']}")
    print(f"  File size: {stage2_path.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict, Counter

stage2_path = Path(DATA_DIR) / "gui-model_stage2.jsonl"

# === 1. 전체 데이터 로드 ===
with open(stage2_path, 'r') as f:
    entries = [json.loads(line) for line in f if line.strip()]

print(f"[Load] Total entries: {len(entries)}")

# === 2. Action type별 그룹화 (Stratified) ===
type_groups = defaultdict(list)
for entry in entries:
    action = json.loads(entry['messages'][-1]['value'])
    action_type = action.get('type', 'unknown')
    type_groups[action_type].append(entry)

print(f"[Group] Action types: {dict(Counter({k: len(v) for k, v in type_groups.items()}))}")

# === 3. Stratified Split (0.9:0.1) ===
random.seed(42)
train_entries, test_entries = [], []

for action_type, group in type_groups.items():
    random.shuffle(group)
    split_idx = max(1, int(len(group) * 0.9))  # 최소 1개는 test에
    train_entries.extend(group[:split_idx])
    test_entries.extend(group[split_idx:])

# 셔플
random.shuffle(train_entries)
random.shuffle(test_entries)

# === 4. 저장 ===
train_path = Path(DATA_DIR) / "gui-model_stage2_train.jsonl"
test_path = Path(DATA_DIR) / "gui-model_stage2_test.jsonl"

for path, data in [(train_path, train_entries), (test_path, test_entries)]:
    with open(path, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# === 5. 검증: action type 비율 비교 ===
print(f"\n{'='*60}")
print(f"Total: {len(entries)} -> Train: {len(train_entries)}, Test: {len(test_entries)}")
print(f"Split ratio: {len(train_entries)/len(entries):.3f} : {len(test_entries)/len(entries):.3f}")

for split_name, split_data in [("Train", train_entries), ("Test", test_entries)]:
    types = Counter()
    for e in split_data:
        a = json.loads(e['messages'][-1]['value'])
        types[a['type']] += 1
    print(f"\n[{split_name}] Action Type Distribution:")
    for t, c in types.most_common():
        print(f"  {t}: {c} ({c/len(split_data)*100:.1f}%)")

In [ ]:
import json
import shutil
from pathlib import Path

DATA_PATH = Path(DATA_DIR)
LF_DATA_DIR = Path(LF_ROOT) / "data"
LF_GUI_DATA = LF_DATA_DIR / "GUI-Model"

# === 1. Stage 2 JSONL 복사 (원본 + train + test) ===
for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    src = DATA_PATH / fname
    dst = LF_GUI_DATA / fname
    if src.exists():
        shutil.copy2(src, dst)
        with open(src, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[Copy] {fname} ({count} entries, {src.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"[WARN] Not found: {src}")

# === 2. dataset_info.json에 Stage 2 등록 (train + test) ===
dataset_info_path = LF_DATA_DIR / "dataset_info.json"
with open(dataset_info_path, 'r', encoding='utf-8') as f:
    dataset_info = json.load(f)

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

STAGE2_DATASETS = {
    "GUI-Model_stage2_train": {
        "file_name": "GUI-Model/gui-model_stage2_train.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    },
    "GUI-Model_stage2_test": {
        "file_name": "GUI-Model/gui-model_stage2_test.jsonl",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS
    }
}

for name, config in STAGE2_DATASETS.items():
    dataset_info[name] = config
    print(f"[Registered] {name}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"\n=== dataset_info.json saved ({len(dataset_info)} datasets) ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter

LF_GUI_DATA = Path(LF_ROOT) / "data" / "GUI-Model"

print("=== Stage 2 Dataset Statistics ===\n")

for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
    fpath = LF_GUI_DATA / fname
    if fpath.exists():
        with open(fpath, 'r') as f:
            lines = f.readlines()

        print(f"[{fname}]")
        print(f"  Entries: {len(lines)}")
        print(f"  File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

        # Action type 분포
        action_types = Counter()
        for line in lines:
            entry = json.loads(line)
            gpt_val = json.loads(entry['messages'][-1]['value'])
            action_types[gpt_val.get('type', 'unknown')] += 1

        print(f"  Action Type Distribution:")
        for atype, count in action_types.most_common():
            print(f"    {atype}: {count} ({count/len(lines)*100:.1f}%)")
        print()
    else:
        print(f"[WARN] Not found: {fpath}")

# 이미지 통계
img_dir = LF_GUI_DATA / "images"
if img_dir.exists():
    imgs = list(img_dir.glob("*.png"))
    total_size = sum(i.stat().st_size for i in imgs) / 1024 / 1024 / 1024
    print(f"[Images]")
    print(f"  Count: {len(imgs)}")
    print(f"  Total size: {total_size:.2f} GB")

## 2. Stage 1 SFT Training

Qwen3-VL-8B-Instruct를 Stage 1 (World Modeling) 데이터로 Full Fine-tuning합니다.
DeepSpeed ZeRO-3 필수 (2x A100 이상 권장).

| 항목 | 값 |
|------|-----|
| Base Model | Qwen/Qwen3-VL-8B-Instruct |
| Method | Full (all parameters) |
| Dataset | GUI-Model_stage1_NoOpenApp_train (2,073건) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 8,192 |
| Epochs | 3.0 |
| Learning Rate | 2e-5 |
| LR Scheduler | constant |
| Warmup | 0.0 |
| Effective Batch | 16 (1 x 8 x 2 GPU) |
| Precision | bf16 |
| Vision Tower | Frozen |
| DeepSpeed | ZeRO-3 |
| Gradient Checkpointing | Enabled |
| Weight Decay | 0.01 |
| Hardware | A100 80GB x 2 |
| Save Steps | 50 |
| Eval Steps | 50 |

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage1_full"
yaml_dir.mkdir(parents=True, exist_ok=True)

STAGE1_CONFIG = """\
### model
model_name_or_path: Qwen/Qwen3-VL-8B-Instruct
trust_remote_code: true
image_max_pixels: 4194304

### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: true

### dataset
dataset: GUI-Model_stage1_NoOpenApp_train
template: qwen3_vl_nothink
cutoff_len: 8192
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/stage1_full
logging_steps: 1
save_steps: 50
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 3.0
lr_scheduler_type: constant
warmup_ratio: 0.0
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
deepspeed: examples/deepspeed/ds_z3_config.json

# ### eval
# val_size: 0.1
# per_device_eval_batch_size: 1
# eval_strategy: steps
# eval_steps: 50
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## [1] Full Fine-tuning (constant, lr=2e-5, DeepSpeed Z3)
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage1_full/qwen3_vl_8b_gui.yaml

## 3. Stage 1 Merge & Upload to HuggingFace

학습된 Stage 1 LoRA 어댑터를 기본 모델에 병합(merge)하고 HuggingFace Hub에 업로드합니다.

**업로드 대상:** `SaFD-00/qwen3-vl-8b-gui`

> **주의:** Merge 시 quantized 모델이나 quantization_bit를 사용하지 마세요.

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "gui"
yaml_dir.mkdir(parents=True, exist_ok=True)

MERGE_STAGE1_CONFIG = """\
### model
model_name_or_path: ./outputs/stage1_full
trust_remote_code: true
template: qwen3_vl_nothink

### export
export_dir: ./exports/qwen3-vl-8b-gui
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: SaFD-00/qwen3-vl-8b-gui
"""

fpath = yaml_dir / "qwen3_vl_8b_gui.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE1_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")

In [ ]:
os.chdir(LF_ROOT)

## LoRA Merge & HuggingFace Hub Upload
!llamafactory-cli export examples/merge_custom/GUI-Model/gui/qwen3_vl_8b_gui.yaml

## 4. Stage 2 SFT Training - World Model 사전학습 효과 검증

**핵심 가설**: GUI World Modeling으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT에서 더 빠르게 수렴할 것이다.

### 3-Way 비교 설계

| ID | 역할 | 모델 | HuggingFace ID |
|----|------|------|----------------|
| [S2-1] | Baseline | Qwen3-VL-8B-Instruct | `Qwen/Qwen3-VL-8B-Instruct` |
| [S2-2] | World Model A | Code2World-8B | `GD-ML/Code2World` |
| [S2-3] | World Model B | gWorld-8B | `trillionlabs/gWorld-8B` |

> 3개 모두 동일한 베이스 모델(Qwen3-VL-8B-Instruct)에서 파생. 공정한 비교 가능.

**공통 학습 설정:**

| 항목 | 값 |
|------|-----|
| Dataset | GUI-Model_stage2_train (~3,290건, 90% split) |
| Method | LoRA (rank=16, alpha=32, target=all, dropout=0.1) |
| Template | qwen3_vl_nothink |
| Cutoff Length | 4,096 |
| Epochs | 3.0 |
| Learning Rate | 5e-5 (cosine, warmup=0.05) |
| Effective Batch | 16 (4 x 2 x 2 GPU) |
| Weight Decay | 0.01 |
| Logging Steps | 1 |
| Precision | bf16 |
| Vision Tower | Frozen |
| Hardware | A100 80GB x 2 |

In [ ]:
import os
from pathlib import Path

# === Stage 2 YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_lora"
yaml_dir.mkdir(parents=True, exist_ok=True)

# 공통 설정 (model_name_or_path, output_dir만 다름)
# 연구 기반 하이퍼파라미터 (Qwen-GUI-3B, ShowUI, LR Matters 논문 참고)
COMMON_CONFIG = """\
### model
model_name_or_path: {model_name_or_path}
trust_remote_code: true
image_max_pixels: 4194304

### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: true
lora_rank: 16
lora_alpha: 32
lora_target: all
lora_dropout: 0.1

### dataset
dataset: GUI-Model_stage2_train
template: qwen3_vl_nothink
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
logging_steps: 1
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
learning_rate: 5.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
weight_decay: 0.01
bf16: true
gradient_checkpointing: true
"""

MODELS = {
    "baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "output_dir": "stage2_baseline",
    },
    "code2world.yaml": {
        "model_name_or_path": "GD-ML/Code2World",
        "output_dir": "stage2_code2world",
    },
    "gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "output_dir": "stage2_gworld",
    },
}

for fname, params in MODELS.items():
    content = COMMON_CONFIG.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  model: {params['model_name_or_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Stage 2 - Baseline (Qwen3-VL-8B-Instruct) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/baseline.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Stage 2 - Code2World-8B (World Model A) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/code2world.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-3] Stage 2 - gWorld-8B (World Model B) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_lora/gworld.yaml

## 5. Stage 2 Evaluation Pipeline

Test set(`GUI-Model_stage2_test`)을 사용하여 3개 모델의 Action Prediction 성능을 비교합니다.

**파이프라인:**
1. Prediction YAML 생성 (3모델)
2. `do_predict`로 test set prediction 수행
3. 평가 함수로 action 단위 메트릭 계산
4. 모델별 비교 테이블 & 시각화

**평가 메트릭:**

| Metric | Formula | Description |
|--------|---------|-------------|
| Parse Rate | 유효 JSON / 전체 | 모델 출력 파싱 성공률 |
| Type Accuracy | 정확 type / 전체 | Action type 일치율 |
| Bounds IoU | IoU(GT, Pred) | Bounding box 겹침 비율 |
| Params Match | 정확 params / 전체 | 파라미터 정확도 |
| Overall Score | Type_Acc × (0.5×IoU + 0.5×Params) | 종합 점수 |

In [ ]:
from pathlib import Path

# === Prediction YAML 설정 파일 생성 ===
yaml_dir = Path(LF_ROOT) / "examples" / "train_custom" / "GUI-Model" / "stage2_predict"
yaml_dir.mkdir(parents=True, exist_ok=True)

PREDICT_TEMPLATE = """\
### model
model_name_or_path: {model_name_or_path}
adapter_name_or_path: {adapter_path}
trust_remote_code: true
finetuning_type: lora
template: qwen3_vl_nothink
image_max_pixels: 4194304

### method
stage: sft
do_predict: true

### dataset
eval_dataset: GUI-Model_stage2_test
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: ./outputs/{output_dir}
overwrite_output_dir: true

### predict
per_device_eval_batch_size: 1
predict_with_generate: true
"""

PREDICT_MODELS = {
    "predict_baseline.yaml": {
        "model_name_or_path": "Qwen/Qwen3-VL-8B-Instruct",
        "adapter_path": "./outputs/stage2_baseline",
        "output_dir": "stage2_predict_baseline",
    },
    "predict_code2world.yaml": {
        "model_name_or_path": "GD-ML/Code2World",
        "adapter_path": "./outputs/stage2_code2world",
        "output_dir": "stage2_predict_code2world",
    },
    "predict_gworld.yaml": {
        "model_name_or_path": "trillionlabs/gWorld-8B",
        "adapter_path": "./outputs/stage2_gworld",
        "output_dir": "stage2_predict_gworld",
    },
}

for fname, params in PREDICT_MODELS.items():
    content = PREDICT_TEMPLATE.format(**params)
    fpath = yaml_dir / fname
    with open(fpath, 'w') as f:
        f.write(content)
    print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
    print(f"  model: {params['model_name_or_path']}")
    print(f"  adapter: {params['adapter_path']}")
    print(f"  output: ./outputs/{params['output_dir']}")
    print()

In [ ]:
os.chdir(LF_ROOT)

## [S2-1] Prediction - Baseline (Qwen3-VL-8B-Instruct) | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_predict/predict_baseline.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-2] Prediction - Code2World-8B | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_predict/predict_code2world.yaml

In [ ]:
os.chdir(LF_ROOT)

## [S2-3] Prediction - gWorld-8B | A100 x 2
!FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=2 \
    llamafactory-cli train examples/train_custom/GUI-Model/stage2_predict/predict_gworld.yaml

In [ ]:
import json
import re
from collections import defaultdict

# ============================================================
# === Action Evaluation Functions ===
# ============================================================

def parse_bounds(bounds_str):
    """'[x1,y1][x2,y2]' -> (x1, y1, x2, y2) or None"""
    if not bounds_str:
        return None
    match = re.findall(r'\[(\d+),(\d+)\]', str(bounds_str))
    if len(match) == 2:
        return (int(match[0][0]), int(match[0][1]),
                int(match[1][0]), int(match[1][1]))
    return None

def calc_iou(box1, box2):
    """두 bounding box의 IoU 계산"""
    if box1 is None or box2 is None:
        return 0.0
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def parse_action(text):
    """모델 출력 텍스트에서 action JSON 파싱"""
    if not text or not text.strip():
        return None
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        # JSON 블록 추출 시도
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return None

def evaluate_single(gt_action, pred_action):
    """단일 예측 평가 -> dict of metrics"""
    result = {
        'type_correct': False,
        'bounds_iou': 0.0,
        'params_match': False,
        'parse_success': pred_action is not None,
        'gt_type': gt_action['type'],
        'pred_type': pred_action.get('type', 'PARSE_FAIL') if pred_action else 'PARSE_FAIL',
    }

    if not pred_action:
        return result

    # 1. Type Accuracy (case-insensitive)
    gt_type = gt_action['type'].lower().strip()
    pred_type = pred_action.get('type', '').lower().strip()
    result['type_correct'] = (gt_type == pred_type)

    # 2. Bounds IoU
    gt_bounds = parse_bounds(gt_action.get('bounds'))
    pred_bounds = parse_bounds(pred_action.get('bounds'))
    if gt_bounds and pred_bounds:
        result['bounds_iou'] = calc_iou(gt_bounds, pred_bounds)
    elif gt_bounds is None and pred_bounds is None:
        result['bounds_iou'] = 1.0  # 둘 다 null이면 정답

    # 3. Params Match
    gt_params = gt_action.get('params', {})
    pred_params = pred_action.get('params', {})
    if gt_type in ['click', 'long click']:
        result['params_match'] = True  # params 없음
    elif gt_type == 'openapp':
        result['params_match'] = (
            gt_params.get('app', '').lower() ==
            pred_params.get('app', '').lower()
        )
    elif gt_type == 'input':
        result['params_match'] = (
            gt_params.get('text', '') == pred_params.get('text', '')
        )
    elif gt_type == 'swipe':
        result['params_match'] = (
            gt_params.get('direction', '').lower() ==
            pred_params.get('direction', '').lower()
        )
    else:
        result['params_match'] = (gt_params == pred_params)

    return result

def evaluate_predictions(gt_path, pred_path):
    """전체 prediction 평가"""
    with open(gt_path) as f:
        gt_entries = [json.loads(l) for l in f if l.strip()]
    with open(pred_path) as f:
        pred_entries = [json.loads(l) for l in f if l.strip()]

    assert len(gt_entries) == len(pred_entries), \
        f"Mismatch: GT={len(gt_entries)}, Pred={len(pred_entries)}"

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('response', ''))
        pred_action = parse_action(pred_text)
        results.append(evaluate_single(gt_action, pred_action))

    # === Aggregate ===
    n = len(results)
    metrics = {
        'total': n,
        'parse_rate': sum(r['parse_success'] for r in results) / n,
        'type_accuracy': sum(r['type_correct'] for r in results) / n,
        'avg_bounds_iou': sum(r['bounds_iou'] for r in results) / n,
        'params_accuracy': sum(r['params_match'] for r in results) / n,
    }

    # Conditional metrics (type이 맞을 때만)
    type_correct = [r for r in results if r['type_correct']]
    if type_correct:
        metrics['cond_bounds_iou'] = sum(r['bounds_iou'] for r in type_correct) / len(type_correct)
        metrics['cond_params_acc'] = sum(r['params_match'] for r in type_correct) / len(type_correct)
    else:
        metrics['cond_bounds_iou'] = 0.0
        metrics['cond_params_acc'] = 0.0

    # Overall Score = Type_Acc * (0.5*IoU + 0.5*Params)
    metrics['overall_score'] = metrics['type_accuracy'] * (
        0.5 * metrics['cond_bounds_iou'] + 0.5 * metrics['cond_params_acc']
    )

    # === Per-type breakdown ===
    type_metrics = defaultdict(lambda: {'correct': 0, 'total': 0, 'iou_sum': 0.0, 'params_sum': 0})
    for r in results:
        tm = type_metrics[r['gt_type']]
        tm['total'] += 1
        tm['correct'] += int(r['type_correct'])
        tm['iou_sum'] += r['bounds_iou']
        tm['params_sum'] += int(r['params_match'])

    metrics['per_type'] = {}
    for t, m in type_metrics.items():
        metrics['per_type'][t] = {
            'count': m['total'],
            'type_acc': m['correct'] / m['total'],
            'avg_iou': m['iou_sum'] / m['total'],
            'params_acc': m['params_sum'] / m['total'],
        }

    return metrics, results

print("[Loaded] Action evaluation functions")
print("  - parse_bounds(), calc_iou(), parse_action()")
print("  - evaluate_single(), evaluate_predictions()")

In [ ]:
import json
import pandas as pd
from pathlib import Path

test_path = Path(LF_ROOT) / "data" / "GUI-Model" / "gui-model_stage2_test.jsonl"

MODEL_PREDS = {
    "Baseline (Qwen3-VL-8B)": Path(LF_ROOT) / "outputs" / "stage2_predict_baseline" / "generated_predictions.jsonl",
    "Code2World-8B": Path(LF_ROOT) / "outputs" / "stage2_predict_code2world" / "generated_predictions.jsonl",
    "gWorld-8B": Path(LF_ROOT) / "outputs" / "stage2_predict_gworld" / "generated_predictions.jsonl",
}

all_metrics = {}
all_results = {}

for model_name, pred_path in MODEL_PREDS.items():
    if not pred_path.exists():
        print(f"[SKIP] {model_name}: {pred_path.name} not found")
        continue
    metrics, results = evaluate_predictions(str(test_path), str(pred_path))
    all_metrics[model_name] = metrics
    all_results[model_name] = results
    print(f"[Done] {model_name}")

# === Summary Table ===
if all_metrics:
    summary_df = pd.DataFrame({
        name: {
            'Parse Rate': f"{m['parse_rate']:.1%}",
            'Type Accuracy': f"{m['type_accuracy']:.1%}",
            'Bounds IoU': f"{m['avg_bounds_iou']:.3f}",
            'Params Accuracy': f"{m['params_accuracy']:.1%}",
            'Cond. IoU': f"{m['cond_bounds_iou']:.3f}",
            'Cond. Params': f"{m['cond_params_acc']:.1%}",
            'Overall Score': f"{m['overall_score']:.3f}",
        }
        for name, m in all_metrics.items()
    }).T

    print("\n" + "="*70)
    print("=== Stage 2 Model Comparison ===")
    print("="*70)
    display(summary_df)

    # === Per-Type Table ===
    for model_name, m in all_metrics.items():
        print(f"\n--- {model_name} Per-Type ---")
        type_df = pd.DataFrame(m['per_type']).T
        type_df.columns = ['Count', 'Type Acc', 'Avg IoU', 'Params Acc']
        display(type_df)
else:
    print("[WARN] No prediction results found. Run prediction cells first.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = list(all_metrics.keys())
metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# === 1. Bar Chart: 모델별 메트릭 비교 ===
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, name in enumerate(model_names):
    values = [all_metrics[name][m] for m in metrics_to_plot]
    axes[0].bar(x + i * width, values, width, label=name, color=colors[i])

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Stage 2: Model Comparison')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metric_labels, rotation=15)
axes[0].legend()
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', alpha=0.3)

# === 2. Per-Type Accuracy Comparison ===
action_types = sorted(set(
    t for m in all_metrics.values() for t in m['per_type'].keys()
))
x2 = np.arange(len(action_types))

for i, name in enumerate(model_names):
    values = [all_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
    axes[1].bar(x2 + i * width, values, width, label=name, color=colors[i])

axes[1].set_xlabel('Action Type')
axes[1].set_ylabel('Type Accuracy')
axes[1].set_title('Per-Type Accuracy Comparison')
axes[1].set_xticks(x2 + width)
axes[1].set_xticklabels(action_types, rotation=30)
axes[1].legend()
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'stage2_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n[Saved] {os.path.join(BASE_DIR, 'stage2_evaluation.png')}")

## 6. Stage 2 Merge & Upload

Stage 2에서 최적의 결과를 보인 모델의 LoRA 어댑터를 병합하고 업로드합니다.

> 학습 결과를 비교한 후, 아래 merge YAML의 `adapter_name_or_path`를 해당 모델의 checkpoint 경로로 수정하세요.

In [ ]:
from pathlib import Path

yaml_dir = Path(LF_ROOT) / "examples" / "merge_custom" / "GUI-Model" / "stage2"
yaml_dir.mkdir(parents=True, exist_ok=True)

# === 최적 모델 설정 (결과 비교 후 수정) ===
BEST_BASE_MODEL = "Qwen/Qwen3-VL-8B-Instruct"  # 또는 GD-ML/Code2World, trillionlabs/gWorld-8B
BEST_ADAPTER_PATH = "./outputs/stage2_baseline"   # 또는 stage2_code2world, stage2_gworld
EXPORT_HUB_ID = "SaFD-00/qwen3-vl-8b-gui-stage2"

MERGE_STAGE2_CONFIG = f"""\
### model
model_name_or_path: {BEST_BASE_MODEL}
adapter_name_or_path: {BEST_ADAPTER_PATH}
trust_remote_code: true
template: qwen3_vl_nothink
finetuning_type: lora

### export
export_dir: ./exports/stage2_best
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {EXPORT_HUB_ID}
"""

fpath = yaml_dir / "best_model.yaml"
with open(fpath, 'w') as f:
    f.write(MERGE_STAGE2_CONFIG)
print(f"[Created] {fpath.relative_to(Path(LF_ROOT))}")
print(f"  Base: {BEST_BASE_MODEL}")
print(f"  Adapter: {BEST_ADAPTER_PATH}")

In [ ]:
os.chdir(LF_ROOT)

## Stage 2 Best Model - LoRA Merge & Upload
## 위 셀에서 BEST_BASE_MODEL, BEST_ADAPTER_PATH를 최적 모델로 수정 후 실행하세요
!llamafactory-cli export examples/merge_custom/GUI-Model/stage2/best_model.yaml